# Week 1 Day 4 Challenge - fewer tokens, the same accuracy

The challenge: find new ways to shorten the prompt that keep the accuracy, and a way to get high accuracy with as few tokens as possible.

The lab already cut Day 2's prompt to the short wording, the sections each claim needs and compact JSON. This notebook starts from the lab's prompt and cuts further in three steps, then tries a different split of the work:

| Variant | What it changes |
|---|---|
| A. Lab | The lab's prompt, unchanged |
| B. This plan only | The limits for the customer's plan and this benefit, in place of the whole limits table; only the fields of the policy record and the claim a decision uses |
| C. Terse | B with shorter instructions and a one-sentence rationale, to cut the output too |
| D. Code pays | C without the Payment section and the limits: the model decides and lists the covered lines, and Python works out the payout |

Each variant runs on three models, three times each, on the lab's 20 claims: the 25 in the queue without the 5 manual ones. I compare the tokens in and out, the cost, the time and the accuracy, on the decision and on the decision with the payout.

## The models

All three are on OpenRouter:

- DeepSeek V4.1 Flash at `low` reasoning, as in the lab, and the product's model
- GPT-6 Luna at `low` reasoning, the cheapest paid model that passed my Week 1 Day 2 challenge
- Nemotron 3.5 Super, a free model, at `low` reasoning

Nemotron 3 Ultra 550B, the free model that passed my Week 1 Day 2 challenge, has no free endpoint with Structured Outputs, so OpenRouter refuses the lab's call. Nemotron 3.5 Super is the free model from Nvidia that has one.

Nemotron thinks for a long time even at `low`, often a minute or more on one claim, so it decides eight claims at a time; at one call a claim that stays well under the 20 requests a minute that free models allow. The paid models decide five at a time, and the three models run side by side.

For the stretch challenges: I moved the product to local models on Week 1 Day 5, when its harness started to take `ollama:` models; its model comparison has Qwen 3.5 4B on Ollama. I didn't try RunPod.

In [ ]:
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import openai
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# This notebook sits two folders below the repo's root, and the payout rules are in labs/
ROOT = Path("../..")
sys.path.append(str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, work_out_payout  # noqa: E402

In [ ]:
load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## The claims

The lab's 20 claims, with the handlers' decisions.

In [ ]:
DATA = ROOT / "data"


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


labels = {decision["claim_id"]: decision for decision in load("evals/decisions.json")}
claims = [claim for claim in load("claims.json") if labels[claim["claim_id"]]["difficulty"] != "manual"]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

print(f"{len(claims)} claims")

## A. The lab's prompt

The instructions, the sections of the short wording and the compact JSON are the lab's, unchanged.

In [ ]:
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")


def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)

BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits, its benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim["policy_id"]]
    record = {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def lab_messages(claim):
    """Variant A: the lab's prompt."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

## B. This plan only

The lab's prompt sends the whole limits table: ten rows for three plans, when a claim needs two or three numbers from one column. Variant B reads the table from the wording and keeps only the customer's plan and the rows for the claim's benefit. The plan's name goes in that line, so the policy record drops it.

It also drops what the decision doesn't use: the claim's id, its one-line summary (the story says it all), its status, and from the policy record the plan, the ids, where it was sold and its status.

In [ ]:
def read_limits(section):
    """Read the limits table into a dict: for each row, the limit on each plan."""
    lines = [line for line in section.splitlines() if line.startswith("|")]
    plans = [cell.strip().lower() for cell in lines[0].strip("|").split("|")[1:]]
    limits = {}
    for line in lines[2:]:
        cells = [cell.strip() for cell in line.strip("|").split("|")]
        limits[cells[0]] = dict(zip(plans, cells[1:]))
    return limits


LIMITS = read_limits(sections["Limits"])

LIMIT_ROWS = {
    "trip_cancellation": ["Cancellation and Interruption maximum"],
    "trip_interruption": ["Cancellation and Interruption maximum"],
    "trip_delay": ["Trip Delay waiting period, hours", "Trip Delay maximum"],
    "baggage_delay": ["Baggage Delay waiting period, hours", "Baggage Delay maximum"],
    "baggage": ["Baggage maximum", "Per-item limit", "Deductible"],
    "passport": ["Passport maximum"],
    "emergency_medical": ["Emergency Medical maximum", "Deductible"],
}


def plan_limits(claim, rows):
    """The customer's plan and its limits for the claim's benefit, on one line."""
    plan = policies[claim["policy_id"]]["plan"]
    values = "; ".join(f"{row} {LIMITS[row][plan]}" for row in rows)
    return f"Plan: {plan.title()}. Limits in US dollars: {values}."


def compact_prompt(claim):
    """Only the fields of the policy record and the claim a decision uses, as compact JSON."""
    policy = policies[claim["policy_id"]]
    record = {key: policy[key] for key in ["add_ons", "purchased_on", "trip", "travelers"]}
    details = {key: claim[key] for key in ["claim_type", "submitted_on", "incident_date", "incident_location", "story", "items", "amount_claimed"]}
    details["documents"] = [f"{document['name']} ({document['type']})" for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def plan_wording(claim, headings_after):
    """The plan's limits line, then the claim's benefit, the add-ons if the policy has any, and the given sections."""
    headings = [BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += headings_after
    return "\n\n".join([plan_limits(claim, LIMIT_ROWS[claim["claim_type"]])] + [sections[heading].strip() for heading in headings])


def plan_messages(claim):
    """Variant B: the lab's instructions, this plan's limits only, and the fields a decision uses."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{plan_wording(claim, ['Never covered', 'Payment', 'Claims'])}\n"},
        {"role": "user", "content": compact_prompt(claim)},
    ]

## C. Terse

The same wording and claim as B, with the instructions cut to what they must say, and a rationale of one sentence. The rationale still comes before the decision, so the model works through the claim first, but a reasoning model does that thinking anyway, so the long rationale is mostly output tokens to pay for.

In [ ]:
TERSE_INSTRUCTIONS = """Decide this UnderCover Travel Insurance claim from the wording, the policy record and the claim.
- approve: covered, with the required documents. Pay as the Payment section says.
- decline: not covered.
- refer: a required document is missing, the story and the documents disagree, or the wording doesn't settle it.
"""


class TerseDecision(BaseModel):
    """A decision on one claim."""

    rationale: str = Field(description="One sentence, naming the section that decides it")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="US dollars, 0 unless approved")


def terse_messages(claim):
    """Variant C: B with shorter instructions."""
    return [
        {"role": "system", "content": f"{TERSE_INSTRUCTIONS}\n{plan_wording(claim, ['Never covered', 'Payment', 'Claims'])}\n"},
        {"role": "user", "content": compact_prompt(claim)},
    ]

## D. Code pays

The payout is arithmetic: the per-item limit, what others paid, the deductible, the maximum. A model doesn't need the Payment section or the money limits to do it if code does it instead. Variant D sends only the waiting periods, which decide whether a delay is covered, and asks the model for the decision, the benefit and the covered lines. Then `work_out_payout`, the payout code from the Week 2 Day 1 lab, works out the payout from the customer's policy.

The lines come back in a schema with a field for each thing the payout code needs, and the schema counts as input tokens too, so this variant trades some of the wording it saves for the schema.

In [ ]:
WAITING_ROWS = {
    "trip_delay": ["Trip Delay waiting period, hours"],
    "baggage_delay": ["Baggage Delay waiting period, hours"],
}

CODE_PAYS_INSTRUCTIONS = """Decide this UnderCover Travel Insurance claim from the wording, the policy record and the claim.
- approve: covered, with the required documents. List the covered items and expenses; leave out any the wording excludes. Code works out the payout.
- decline: not covered.
- refer: a required document is missing, the story and the documents disagree, or the wording doesn't settle it.
"""


class CoveredClaim(BaseModel):
    """A decision on one claim, with what is covered."""

    rationale: str = Field(description="One sentence, naming the section that decides it")
    decision: Literal["approve", "decline", "refer"]
    benefit: Benefit = Field(description="The benefit the claim is paid under")
    lines: list[Line] = Field(description="For an approval, the covered items and expenses, one for each line of the claim; empty otherwise")
    paid_by_others: float = Field(description="Anything a carrier, hotel or other party paid or refunded for the same loss, or 0")


def code_pays_wording(claim):
    """The waiting period if the benefit has one, then the claim's benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = [BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Claims"]
    parts = [sections[heading].strip() for heading in headings]
    if claim["claim_type"] in WAITING_ROWS:
        parts.insert(0, plan_limits(claim, WAITING_ROWS[claim["claim_type"]]))
    return "\n\n".join(parts)


def code_pays_messages(claim):
    """Variant D: no Payment section and no money limits; the model lists the covered lines."""
    return [
        {"role": "system", "content": f"{CODE_PAYS_INSTRUCTIONS}\n{code_pays_wording(claim)}\n"},
        {"role": "user", "content": compact_prompt(claim)},
    ]

Here's the same claim in all four variants, as the model sees it.

In [ ]:
VARIANTS = {
    "A. Lab": {"messages": lab_messages, "schema": ClaimDecision},
    "B. This plan only": {"messages": plan_messages, "schema": ClaimDecision},
    "C. Terse": {"messages": terse_messages, "schema": TerseDecision},
    "D. Code pays": {"messages": code_pays_messages, "schema": CoveredClaim},
}

for name, variant in VARIANTS.items():
    messages = variant["messages"](claims[0])
    words = sum(len(message["content"].split()) for message in messages)
    print(f"===== {name}: {words} words =====\n")
    print(messages[0]["content"])
    print(messages[1]["content"])

And the size of each variant's prompt across the 20 claims, in words, before any model sees it. The schema isn't counted here; the input tokens below include it.

In [ ]:
pd.DataFrame(
    {name: [sum(len(message["content"].split()) for message in variant["messages"](claim)) for claim in claims] for name, variant in VARIANTS.items()}
).describe().loc[["mean", "min", "max"]].round(0)

## Deciding a claim

`decide` is the lab's function, for any variant: it sends the variant's prompt with its schema through Structured Outputs. For variant D it works out the payout with `work_out_payout`. OpenRouter reports the cost of each call.

A claim stopped by a rate limit waits a minute and starts again, up to three tries. Any other failure, such as a reply that doesn't fit the schema, counts the claim as wrong and keeps the error.

In [ ]:
MODELS = {
    "DeepSeek V4.1 Flash": {"model": "deepseek/deepseek-v4.1-flash", "effort": "low", "claims_at_a_time": 5},
    "GPT-6 Luna": {"model": "openai/gpt-6-luna", "effort": "low", "claims_at_a_time": 5},
    "Nemotron 3.5 Super (free)": {"model": "nvidia/nemotron-3.5-super-vl:free", "effort": "low", "claims_at_a_time": 8},
}

RATE_LIMIT_TRIES = 3


def is_right(decision, payout, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return decision == label["decision"] and abs(payout - label["payout"]) <= 1


def ask(claim, model_name, variant_name):
    """Send one claim to one model with one variant's prompt and schema, and return the reply."""
    model, variant = MODELS[model_name], VARIANTS[variant_name]
    for attempt in range(RATE_LIMIT_TRIES):
        try:
            return openrouter.chat.completions.parse(
                model=model["model"],
                messages=variant["messages"](claim),
                response_format=variant["schema"],
                reasoning_effort=model["effort"],
                extra_body={"provider": {"require_parameters": True}},
            )
        except openai.RateLimitError:
            if attempt == RATE_LIMIT_TRIES - 1:
                raise
            time.sleep(60)


def decide(claim, model_name, variant_name, run):
    """Decide one claim and score it against the handler's decision."""
    label = labels[claim["claim_id"]]
    result = {"model": model_name, "variant": variant_name, "run": run, "claim_id": claim["claim_id"], "difficulty": label["difficulty"]}
    start = time.time()
    try:
        response = ask(claim, model_name, variant_name)
    except Exception as exception:
        return {**result, "decision": "error", "payout": 0.0, "decision_right": False, "right": False, "error": f"{type(exception).__name__}: {exception}",
                "input_tokens": 0, "output_tokens": 0, "reasoning_tokens": 0, "cost": 0.0, "seconds": round(time.time() - start, 1)}
    answer = response.choices[0].message.parsed
    error = ""
    if answer is None:
        decision, payout, error = "error", 0.0, "The reply didn't fit the schema"
    elif isinstance(answer, CoveredClaim):
        decision = answer.decision
        payout = work_out_payout(policies[claim["policy_id"]], answer.benefit, answer.lines, answer.paid_by_others)["payout"] if decision == "approve" else 0.0
    else:
        decision, payout = answer.decision, answer.payout
    usage = response.usage
    details = usage.completion_tokens_details
    return {
        **result,
        "decision": decision,
        "payout": payout,
        "decision_right": decision == label["decision"],
        "right": is_right(decision, payout, label),
        "error": error,
        "rationale": answer.rationale if answer else "",
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "reasoning_tokens": (details.reasoning_tokens or 0) if details else 0,
        "cost": getattr(usage, "cost", 0.0) or 0.0,
        "seconds": round(time.time() - start, 1),
    }

## Run every variant on every model, three times

Each model works through the variants and runs on its own thread, and each run's claims go to a pool of five or eight threads. Each finished run is saved to `week1day4_runs.json`, so if the notebook stops, running this cell again carries on from the next run. Delete the file to start again.

In [ ]:
N_RUNS = 3
RUNS_FILE = Path("week1day4_runs.json")

results = json.loads(RUNS_FILE.read_text(encoding="utf-8")) if RUNS_FILE.exists() else []
done = {(result["model"], result["variant"], result["run"]) for result in results}
saving = threading.Lock()


def run_model(model_name):
    """Run every variant on one model, N_RUNS times, saving each run as it finishes."""
    for variant_name in VARIANTS:
        for run in range(1, N_RUNS + 1):
            if (model_name, variant_name, run) in done:
                continue
            with ThreadPoolExecutor(max_workers=MODELS[model_name]["claims_at_a_time"]) as pool:
                run_results = list(pool.map(lambda claim: decide(claim, model_name, variant_name, run), claims))
            with saving:
                results.extend(run_results)
                RUNS_FILE.write_text(json.dumps(results, indent=2), encoding="utf-8")


with ThreadPoolExecutor(max_workers=len(MODELS)) as pool:
    list(pool.map(run_model, MODELS))

rank = {name: i for i, name in enumerate([*MODELS, *VARIANTS])}
scored = pd.DataFrame(results).sort_values(
    ["model", "variant", "run", "claim_id"], key=lambda column: column.map(rank) if column.name in ("model", "variant") else column
)

for (model_name, variant_name, run), run_results in scored.groupby(["model", "variant", "run"], sort=False):
    errors = (run_results["error"] != "").sum()
    print(
        f"{model_name}, {variant_name}, run {run}: {run_results['decision_right'].sum()} decisions right, "
        f"{run_results['right'].sum()} with the payout, of {len(run_results)}; {errors} errors, ${run_results['cost'].sum():.4f}"
    )

## The results

For each model and variant: the claims right on the decision and the payout, fewest and mean across the three runs, out of 20; the decisions right on their own; and the tokens, cost and time for one claim.

In [ ]:
runs = scored.groupby(["model", "variant", "run"], sort=False).agg(right=("right", "sum"), decision_right=("decision_right", "sum")).reset_index()
accuracy = runs.groupby(["model", "variant"], sort=False).agg(
    fewest_right=("right", "min"), mean_right=("right", "mean"), mean_decisions_right=("decision_right", "mean")
)
per_claim = scored.groupby(["model", "variant"], sort=False).agg(
    input_tokens=("input_tokens", "mean"),
    output_tokens=("output_tokens", "mean"),
    reasoning_tokens=("reasoning_tokens", "mean"),
    cost_per_claim=("cost", "mean"),
    seconds_per_claim=("seconds", "mean"),
    errors=("error", lambda errors: (errors != "").sum()),
)
summary = accuracy.join(per_claim)
summary[["mean_right", "mean_decisions_right", "seconds_per_claim"]] = summary[["mean_right", "mean_decisions_right", "seconds_per_claim"]].round(1)
summary[["input_tokens", "output_tokens", "reasoning_tokens"]] = summary[["input_tokens", "output_tokens", "reasoning_tokens"]].round(0).astype(int)
summary["cost_per_claim"] = summary["cost_per_claim"].map(lambda cost: f"${cost:.5f}")
summary

### Each variant against the lab's prompt

The same numbers for each variant, summed over the models, as a change from variant A: positive is more, negative is less.

In [ ]:
by_variant = scored.groupby("variant", sort=False).agg(
    right=("right", "mean"), input_tokens=("input_tokens", "mean"), output_tokens=("output_tokens", "mean"), cost=("cost", "mean")
)
change = pd.DataFrame(
    {
        "right (%)": (100 * by_variant["right"]).round(1),
        "input tokens": by_variant["input_tokens"].round(0).astype(int),
        "input vs A (%)": (100 * (by_variant["input_tokens"] / by_variant["input_tokens"].iloc[0] - 1)).round(0),
        "output tokens": by_variant["output_tokens"].round(0).astype(int),
        "output vs A (%)": (100 * (by_variant["output_tokens"] / by_variant["output_tokens"].iloc[0] - 1)).round(0),
        "cost vs A (%)": (100 * (by_variant["cost"] / by_variant["cost"].iloc[0] - 1)).round(0),
    }
)
change

### Accuracy against tokens

Each point is one model on one variant: the mean claims right, decision and payout, against the mean tokens in and out for one claim. The best place is the top left.

In [ ]:
points = summary.reset_index()
points["tokens"] = points["input_tokens"] + points["output_tokens"]
points["right (%)"] = 100 * points["mean_right"] / len(claims)
fig = px.scatter(
    points,
    x="tokens",
    y="right (%)",
    color="model",
    symbol="variant",
    text="variant",
    title="Accuracy against tokens for one claim",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

## The claims each variant got wrong

For each claim, the number of times each variant got it wrong across the three models and three runs, out of nine. Claims every variant got right every time are left out.

In [ ]:
wrong = scored.assign(wrong=~scored["right"])
misses = wrong.pivot_table(index="claim_id", columns="variant", values="wrong", aggfunc="sum", sort=False)[list(VARIANTS)]
misses = misses[misses.sum(axis=1) > 0]
handlers = pd.DataFrame(
    {claim_id: {"claim type": next(claim["claim_type"] for claim in claims if claim["claim_id"] == claim_id), "handler": f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']:,.2f}"} for claim_id in misses.index}
).T
handlers.join(misses)

To see what each model said on one claim with each variant, pass its id to `show_claim`.

In [ ]:
def show_claim(claim_id):
    """Each model's decision and payout on one claim, for each variant and run."""
    label = labels[claim_id]
    print(f"{claim_id}: the handler said {label['decision']} ${label['payout']:,.2f}. {label['reason']}\n")
    rows = scored[scored["claim_id"] == claim_id]
    rows = rows.assign(answer=rows["decision"] + " $" + rows["payout"].map("{:,.2f}".format))
    return rows.pivot_table(index=["model", "variant"], columns="run", values="answer", aggfunc="first", sort=False)


show_claim("CLM-1011")

CLM-1009, where variant D's payout code was only as good as the kind the model gave the item.

In [ ]:
show_claim("CLM-1009")

## What I found

**Variant B is the cut to keep: 27% fewer input tokens, with the same accuracy.** Across the three models, B got 70.6% of the claims right on the decision and the payout, against 69.4% for the lab's prompt. Sending only the customer's plan and the rows for the claim's benefit, and leaving out the fields a decision doesn't use, lost nothing.

**Fewer input tokens didn't make a claim cheaper.** Every shorter variant cost more a claim than the lab's prompt:

| Model | A. Lab | B. This plan only | C. Terse | D. Code pays |
|---|---|---|---|---|
| DeepSeek V4.1 Flash | $0.00080 | $0.00109 | $0.00097 | $0.00101 |
| GPT-6 Luna | $0.00012 | $0.00018 | $0.00015 | $0.00019 |

The models thought for longer on the shorter prompts. DeepSeek's reasoning went from 925 tokens a claim on A to about 1,200 on B, C and D, and Luna's from 45 to between 76 and 140. Output tokens are where the money goes: DeepSeek's cost 50 times its input tokens, and most of its output is reasoning. Saving 300 input tokens doesn't pay for 300 more tokens of thinking. On these models the next thing to try is the reasoning setting, not the prompt.

**C, the terse instructions, is a cut too far for Luna.** Luna's decisions right fell from 14.7 of 20 on B to 12.0 on C, mostly by referring claims it had approved, such as CLM-1011 in two of three runs. DeepSeek and Nemotron did as well on C as on B. How short the instructions can be depends on the model.

**D, code pays, did worse.** The decisions were as good as on A, 17.0 of 20 on both for DeepSeek, but fewer payouts were right: 60.6% of the claims across the models, against 69.4%. Two things went against it:

- The schema for the lines counts as input, so D saved only 15% of the input tokens against A, less than B.
- The model now has to sort each line into the payout code's kinds, and that's where it slipped. CLM-1009 is one $980 item, and on A only one of the nine tries got it wrong. On D, DeepSeek and Nemotron filed it as an ordinary item at the $300 per-item limit in every run, and the code paid $250; Luna filed it as electronics under Electronics Plus, and the code paid the handler's $930. The arithmetic wasn't where the mistakes were, and moving it to code moved them into the sorting.

**Some claims are wrong whatever the prompt.** CLM-1011, CLM-1016, CLM-1023 and CLM-1025 were wrong in 35 or 36 of the 36 tries on each claim. CLM-1011, the lost suitcase, came back $1,500 almost every time, where the handler paid $1,150: the models group the items the wrong way. In its rationales Luna counts the four shirts as one set and the tuxedo's jacket and trousers, and the camera and its lens, as separate items, so the total reaches the maximum; DeepSeek's closest try, $1,370, grouped the tuxedo but not the camera and its lens. Trimming the prompt can't fix a rule the model doesn't apply.

**The models.** DeepSeek V4.1 Flash was the most accurate, with 15.3 claims right on A and 15.7 on C, at about a tenth of a cent a claim. Luna was the cheapest paid model, at about a hundredth of a cent. Nemotron 3.5 Super cost nothing, but took 40 to 54 seconds a claim and failed on 14 of its 240 calls: 8 replies came back with no answer from the provider and 6 didn't fit the schema.

What I'd use: variant B on DeepSeek V4.1 Flash. The whole comparison, 720 calls, cost $0.27.